# Stochastic Calculus from Zero · Episode 12: Martingales made from Brownian motion

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Three martingales · 4. Given the present · 5. The exponential · 6. This episode's question · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 12,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)

## 2. Last episode's answer

The quadratic variation of $3W$ over two years is $3^2 \times 2 = 18$.

In [ ]:
out["prev"] = {"answer": 18}

## 3. Three martingales

For $s < t$, write $W_t = W_s + D$ with $D = W_t - W_s$ independent of the past, mean 0, variance $t - s$.

- $E[W_t \mid \mathcal{F}_s] = W_s + E[D] = W_s$: **$W$ is a martingale**.
- $E[W_t^2 \mid \mathcal{F}_s] = W_s^2 + 2W_s E[D] + E[D^2] = W_s^2 + (t - s)$, so **$W_t^2 - t$ is a martingale** (Part 1's $M_n^2 - n$).
- $E[e^{\sigma W_t} \mid \mathcal{F}_s] = e^{\sigma W_s} E[e^{\sigma D}] = e^{\sigma W_s} e^{\frac12 \sigma^2 (t-s)}$, so **$\exp(\sigma W_t - \tfrac12\sigma^2 t)$ is a martingale**.

All three should average their starting value (0, 0 and 1) at every time. 100,000 paths over two years, $\sigma = 20\%$:

In [ ]:
rng = np.random.default_rng(seed)
sig = 0.20
_, W, _ = brownian_paths(100_000, 200, 2.0, rng)
rows = []
for t_, i in ((0.5, 50), (1.0, 100), (2.0, 200)):
    w = W[:, i]
    rows.append({"t": t_, "W": float(w.mean()), "W2_minus_t": float((w ** 2 - t_).mean()),
                 "exp_mart": float(np.exp(sig * w - 0.5 * sig ** 2 * t_).mean()),
                 "exp_plain": float(np.exp(sig * w).mean()), "exp_plain_theory": math.exp(0.5 * sig ** 2 * t_)})
    se = math.sqrt(t_ / 100_000)
    assert abs(rows[-1]["W"]) < 4 * se and abs(rows[-1]["W2_minus_t"]) < 4 * t_ * math.sqrt(2 / 100_000)
    assert abs(rows[-1]["exp_mart"] - 1) < 0.005
out["by_time"] = {"rows": rows, "sigma": sig}
pd.DataFrame(rows)

## 4. Given the present

Constant averages aren't enough (Part 1, Episode 6): a martingale must keep its best guess *given what is known*. Group the paths by where they are at $t = 0.5$ and, within each group, compare the average of $W_1^2 - 1$ with the average of $W_{0.5}^2 - 0.5$. They agree within three standard errors in every group.

In [ ]:
edges = np.linspace(-2, 2, 9)
grp = np.digitize(W[:, 50], edges)
cond = []
for g in range(1, len(edges)):
    m = grp == g
    fut = W[m, 100] ** 2 - 1
    cond.append({"group": f"{edges[g - 1]:+.1f} to {edges[g]:+.1f}", "now": float((W[m, 50] ** 2 - 0.5).mean()),
                 "later": float(fut.mean()), "se": float(fut.std() / math.sqrt(m.sum())), "paths": int(m.sum())})
    assert abs(cond[-1]["later"] - cond[-1]["now"]) < 3 * cond[-1]["se"]
out["cond"] = {"rows": cond, "groups": len(cond)}
pd.DataFrame(cond)

## 5. The exponential

$e^{\sigma W_t}$ is not a martingale: its average grows like $e^{\frac12 \sigma^2 t}$, because $e^x$ is convex and averages of convex functions exceed the function of the average (Jensen's inequality). Subtracting $\tfrac12 \sigma^2 t$ in the exponent removes exactly that growth. This is the shape of a share price under the risk-neutral odds (Series 3, Episode 13: volatility drag) and of Girsanov's reweighting $Z$ (Part 1, Episode 8; Part 3).

In [ ]:
out["exp"] = {"plain_1": rows[1]["exp_plain"], "theory_1": rows[1]["exp_plain_theory"], "mart_2": rows[2]["exp_mart"],
              "plain_2": rows[2]["exp_plain"], "theory_2": rows[2]["exp_plain_theory"]}
out["exp"]

## 6. This episode's question

What do you subtract from $W_t^3$ to get a martingale? $E[W_t^3 \mid \mathcal{F}_s] = W_s^3 + 3W_s(t - s)$, so the drift of $W^3$ at time $u$ is $3W_u$ per unit time: $W_t^3 - 3\int_0^t W_u\,du$ is a martingale. Episode 15 gets this in one line from Itô's formula.

In [ ]:
drift = 3 * W[:, :-1].sum(axis=1) * (2.0 / 200)            # 3 * integral of W du, left sums
m3 = W[:, -1] ** 3 - drift
out["question"] = {"mean_W3_minus_drift": float(m3.mean()), "se": float(m3.std() / math.sqrt(len(m3)))}
assert abs(out["question"]["mean_W3_minus_drift"]) < 4 * out["question"]["se"]
out["question"]

## 7. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Derive $E[W_t^2 \mid \mathcal{F}_s]$ for $s < t$.

<details><summary>Answer</summary>

$W_t^2 = (W_s + D)^2 = W_s^2 + 2W_s D + D^2$. Given $\mathcal{F}_s$, $W_s$ is known, $E[D] = 0$ and $E[D^2] = t - s$: the answer is $W_s^2 + t - s$.

</details>

**Exercise 2 (By hand).** Use the normal moment generating function $E[e^{aZ}] = e^{a^2 v/2}$ for $Z \sim N(0, v)$ to find $E[e^{\sigma W_t}]$.

<details><summary>Answer</summary>

$e^{\sigma^2 t / 2}$: here $a = \sigma$ and $v = t$.

</details>

**Exercise 3 (By hand).** Is $W_t + t$ a martingale? Is it a sub- or supermartingale?

<details><summary>Answer</summary>

No: $E[W_t + t \mid \mathcal{F}_s] = W_s + t > W_s + s$. It drifts up, so it is a submartingale.

</details>

**Exercise 4 (In code).** Check by simulation that $W_t^3 - 3tW_t$ averages zero at $t = 0.5, 1, 2$. (It is a martingale too.)

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
for t_, i in ((0.5, 50), (1.0, 100), (2.0, 200)):
    v = W[:, i] ** 3 - 3 * t_ * W[:, i]
    assert abs(v.mean()) < 4 * v.std() / math.sqrt(len(v))
    print(f"t = {t_}: mean of W^3 - 3tW = {v.mean():+.4f} (standard error {v.std() / math.sqrt(len(v)):.4f})")

**Exercise 5 (In code).** Repeat Section 4's grouping test for the exponential martingale $\exp(\sigma W_t - \tfrac12\sigma^2 t)$.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
e = lambda w, t_: np.exp(sig * w - 0.5 * sig ** 2 * t_)
for g in range(1, len(edges)):
    m = grp == g
    later, now = e(W[m, 100], 1.0), e(W[m, 50], 0.5)
    assert abs(later.mean() - now.mean()) < 3 * later.std() / math.sqrt(m.sum())
print("exponential martingale: later average matches today's value in every group")

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")